# Aula ML 05 — Séries temporais

A linha tem data. O teste é 2012. A baseline é a mesma hora, sete dias antes.

| slide | célula |
|---|---|
| 3 | o arquivo e janeiro |
| 4 | casual + registered |
| 5 | o corte 2011 / 2012 |
| 7 | a baseline |
| 8 | a reta com calendário |
| 9 | o split aleatório |


## De onde veio o arquivo

[Bike Sharing Demand](https://www.kaggle.com/competitions/bike-sharing-demand), o `train.csv`. A aula lê `dados/train.csv`.

In [ ]:
import kagglehub
kagglehub.competition_download("bike-sharing-demand")


## ▸ Slides 3 e 4 · O arquivo e a coluna que é o alvo

In [ ]:
import os
os.environ.setdefault("MLFLOW_ALLOW_FILE_STORE", "true")

import mlflow
import numpy as np
import pandas as pd
from sklearn.compose import ColumnTransformer
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_squared_error
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler

mlflow.set_tracking_uri("mlruns")
mlflow.set_experiment("aula-ml-5-bicicletas")

horas = pd.read_csv("dados/train.csv", parse_dates=["datetime"]).sort_values("datetime")
assert (horas["casual"] + horas["registered"] == horas["count"]).all()
print(len(horas), "horas · dias", sorted(horas["datetime"].dt.day.unique()))
print(horas[["casual", "registered", "count"]].head(3))


## ▸ Slide 5 · O corte

In [ ]:
horas["hora"] = horas["datetime"].dt.hour
horas["dia_semana"] = horas["datetime"].dt.dayofweek
horas["mes"] = horas["datetime"].dt.month
treino = horas[horas["datetime"].dt.year == 2011]
teste = horas[horas["datetime"].dt.year == 2012]
print(len(treino), "em 2011 ·", len(teste), "em 2012")


## ▸ Slides 7 e 8 · Baseline e reta

In [ ]:
def rmse(a, b):
    return float(np.sqrt(mean_squared_error(a, b)))

teste = teste.copy()
teste["chave"] = teste["datetime"] - pd.Timedelta(days=7)
base = horas[["datetime", "count"]].rename(columns={"datetime": "chave", "count": "lag"})
com_par = teste.merge(base, on="chave", how="left").dropna(subset=["lag"])
print("baseline", round(rmse(com_par["count"], com_par["lag"])), "em", len(com_par), "horas")

cat = ["hora", "dia_semana", "mes", "season", "weather"]
num = ["temp", "humidity", "windspeed"]
binarias = ["holiday", "workingday"]
features = cat + num + binarias
reta = Pipeline([
    ("preparo", ColumnTransformer([
        ("cat", OneHotEncoder(handle_unknown="ignore", sparse_output=False), cat),
        ("num", StandardScaler(), num),
        ("bin", "passthrough", binarias),
    ])),
    ("modelo", LinearRegression()),
])
reta.fit(treino[features], treino["count"])
print("reta 2012", round(rmse(teste["count"], reta.predict(teste[features]))))
print("reta no recorte da baseline", round(rmse(com_par["count"], reta.predict(com_par[features]))))
with mlflow.start_run(run_name="baseline-semana"):
    mlflow.log_metric("teste_rmse", rmse(com_par["count"], com_par["lag"]))
with mlflow.start_run(run_name="reta-calendario"):
    mlflow.log_metric("teste_rmse", rmse(teste["count"], reta.predict(teste[features])))


## ▸ Slide 9 · O split aleatório

In [ ]:
X_treino, X_teste, y_treino, y_teste = train_test_split(
    horas[features], horas["count"], test_size=0.25, random_state=0
)
outro = Pipeline([
    ("preparo", ColumnTransformer([
        ("cat", OneHotEncoder(handle_unknown="ignore", sparse_output=False), cat),
        ("num", StandardScaler(), num),
        ("bin", "passthrough", binarias),
    ])),
    ("modelo", LinearRegression()),
])
outro.fit(X_treino, y_treino)
print("split aleatório", round(rmse(y_teste, outro.predict(X_teste))))
with mlflow.start_run(run_name="reta-split-aleatorio"):
    mlflow.log_metric("teste_sorteado_rmse", rmse(y_teste, outro.predict(X_teste)))
